In [ ]:
import re
import sys
import pandas as pd

sys.path.append("..")
from vocab import terms

df = pd.read_csv("../data/raw/openalex_alan.csv")
df["text"] = df["title"].fillna("") + ". " + df["abstract"].fillna("")


def make_pattern(term):
    # Abbreviations like SQM, PAR, W must match exactly (capital letters).
    case_sensitive = term.isupper() or term == "lx"
    if case_sensitive:
        return re.compile(r"(?<!\w)" + re.escape(term) + r"(?!\w)")
    # Other terms ignore case and also match a plural ("moth" -> "moths").
    return re.compile(r"(?<!\w)" + re.escape(term) + r"(?:e?s)?(?!\w)", re.IGNORECASE)


def find_terms(text, term_list):
    found = [term for term in term_list if make_pattern(term).search(text)]
    return "; ".join(found)

In [ ]:
%%writefile ../vocab/terms.py

SEARCH_TERMS = [
    "artificial light at night", "ALAN", "light pollution",
    "ecological light pollution", "skyglow", "night sky brightness",
    "nocturnal light", "light at night",
]

METHODS_INSTRUMENTS = [
    "Sky Quality Meter", "SQM", "radiometer", "lux meter", "photometer",
    "spectroradiometer", "hyperspectral", "multispectral", "fisheye",
    "digital camera", "DSLR", "all-sky", "all-sky photometry",
    "full-sphere", "differential photometry", "sky scanner", "ASTMON",
    "dark sky unit", "remote sensing", "satellite", "International Space Station",
    "UAV", "drone", "dose-response",
]

QUANTITIES = [
    "illuminance", "irradiance", "radiance", "sky brightness",
    "photosynthetically active radiation", "PAR", "photon flux",
    "spectral composition", "spectral sensitivity", "photoperiod",
    "intensity", "wavelength", "threshold",
]

UNITS = [
    "lux", "lx", "mag/arcsec2", "magnitudes per square arcsecond",
    "µmol/s", "μmol photons", "W", "watts",
]

DISCIPLINES = [
    "astronomy", "physics", "ecology", "chronobiology", "psychology",
    "engineering", "lighting engineering", "visual ecology", "toxicology",
    "photobiology", "radiometry", "photometry", "conservation",
]

TAXA = [
    "insect", "moth", "Lepidoptera", "firefly", "dung beetle", "aphid",
    "parasitoid", "arthropod", "slug", "bat", "bird", "songbird", "great tit",
    "Parus major", "fish", "European perch", "turtle", "sea turtle",
    "loggerhead", "Caretta caretta", "zooplankton", "Daphnia", "coral",
    "crab", "sandhopper", "mammal", "vertebrate", "microbial",
    "plant", "tree", "grassland",
]

In [ ]:
importlib.reload(terms)
print([name for name in dir(terms) if name.isupper()])

In [ ]:
categories = {
    "methods": terms.METHODS_INSTRUMENTS,
    "quantities": terms.QUANTITIES,
    "units": terms.UNITS,
    "disciplines": terms.DISCIPLINES,
    "species": terms.TAXA,
}

for column, term_list in categories.items():
    df[column] = df["text"].apply(lambda t: find_terms(t, term_list))

df[["first_author", "year", "venue", "methods", "quantities", "units", "disciplines", "species"]].head(100)

In [ ]:
out = df.drop(columns=["text"])
out.to_csv("../data/output/table_abstracts.csv", index=False)

for column in categories:
    print(column, "->", (df[column] != "").sum(), "papers with at least one match")

In [ ]:
import pandas as pd

kg = pd.read_csv("../kg/seed_statements.csv")
print(kg.shape)
print(kg.isna().sum())
print((kg.apply(lambda col: col.str.strip() != col)).sum())
print(kg["property"].value_counts())

In [ ]:
print(kg[kg["source"].str.strip() != kg["source"]])
print()
print(sorted(kg["subject"].unique()))